# Peek at MAG-based taxonomy ↔ biosamples

Worked-example queries linking metagenome-assembled genome (MAG) taxonomy and
quality results to biosamples via `nmdc_metadata.biosample_to_workflow_run`.

Covers:
- **GTDBTK bacterial classification** (`nmdc_results.gtdbtk_bacterial_summary`) — full lineage per MAG bin
- **CheckM quality statistics** (`nmdc_results.checkm_statistics`) — completeness, contamination

**Both directions:**
- §1–2: biosample → MAG classifications / quality for that sample
- §4–5: taxonomy substring / quality threshold → all biosamples

**Note:** GTDBTK archaeal (`nmdc_results.gtdbtk_archaeal_summary`) has no data in this corpus —
every file was a single-line placeholder. See §3.

In [ ]:
spark = get_spark_session(app_name="peek_mag_taxonomy_links", tenant_name="nmdc")
print(f"Spark version: {spark.version}")

### Preflight: check available MAG taxonomy tables

In [ ]:
existing = {r["tableName"] for r in spark.sql("SHOW TABLES IN nmdc_results").toPandas().to_dict("records")}

checks = {
    'gtdbtk_bacterial': 'gtdbtk_bacterial_summary',
    'checkm':           'checkm_statistics',
    'gtdbtk_archaeal':  'gtdbtk_archaeal_summary',
}

available = {}
for key, tbl in checks.items():
    found = tbl in existing
    available[key] = found
    status = "OK" if found else "MISSING"
    print(f"{status}: nmdc_results.{tbl}")

### Pick an anchor biosample

Find a biosample that has at least one GTDBTK bacterial result via the workflow bridge.

In [ ]:
anchor = spark.sql("""
    SELECT b2wr.biosample_id, b2wr.workflow_run_id, b2wr.workflow_type
    FROM   nmdc_results.gtdbtk_bacterial_summary g
    JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
             ON b2wr.workflow_run_id = g.workflow_run_id
    LIMIT 1
""").toPandas().iloc[0]
BIOSAMPLE_ID  = anchor["biosample_id"]
GTDBTK_RUN_ID = anchor["workflow_run_id"]
print(anchor.to_string())

### 1. Biosample → GTDBTK bacterial MAG classifications

Each row is one MAG (bin) recovered from this biosample's metagenome assembly.
The `classification` column is a full GTDBTK lineage string.

In [ ]:
spark.sql(f"""
    SELECT g.name AS bin_id, g.classification, g.fastani_reference,
           ROUND(g.fastani_ani, 2) AS fastani_ani, ROUND(g.fastani_af, 2) AS fastani_af
    FROM   nmdc_results.gtdbtk_bacterial_summary g
    WHERE  g.workflow_run_id = '{GTDBTK_RUN_ID}'
    ORDER  BY g.fastani_ani DESC NULLS LAST
""").toPandas()

### 2. Biosample → CheckM MAG quality

Completeness ≥ 90% and contamination ≤ 5% is the standard 'high-quality draft' threshold (MIMAG).

In [ ]:
if not available['checkm']:
    print("SKIP: checkm_statistics not available")
else:
    checkm_run = spark.sql(f"""
        SELECT b2wr.workflow_run_id
        FROM   nmdc_metadata.biosample_to_workflow_run b2wr
        JOIN   nmdc_results.checkm_statistics c ON c.workflow_run_id = b2wr.workflow_run_id
        WHERE  b2wr.biosample_id = '{BIOSAMPLE_ID}'
        LIMIT 1
    """).toPandas()
    if checkm_run.empty:
        print(f"No CheckM results for {BIOSAMPLE_ID}")
    else:
        CHECKM_RUN_ID = checkm_run.iloc[0]["workflow_run_id"]
        spark.sql(f"""
            SELECT bin_id, marker_lineage,
                   ROUND(completeness, 1)        AS completeness,
                   ROUND(contamination, 1)       AS contamination,
                   ROUND(strain_heterogeneity, 1) AS strain_heterogeneity,
                   n_markers
            FROM   nmdc_results.checkm_statistics
            WHERE  workflow_run_id = '{CHECKM_RUN_ID}'
            ORDER  BY completeness DESC, contamination ASC
        """).toPandas()

### 3. GTDBTK archaeal — not available

Every GTDBTK Archaeal file in the NMDC corpus is a single-line placeholder:
```
No Archaeal Results for nmdc:wfmag-...
```
This means none of the 3,535 MAG workflow runs recovered any archaeal bins. Either the
upstream binning pipeline filters out archaeal bins before GTDB-Tk, or none of the
assemblies in this load cycle recovered archaea. The table
`nmdc_results.gtdbtk_archaeal_summary` was not ingested.

When archaeal data becomes available, the query pattern is identical to §1 but
referencing `gtdbtk_archaeal_summary`.

### 4. Taxonomy string → biosamples (GTDBTK)

Find all biosamples that recovered at least one MAG classified to a given taxonomic group.
Filter on a substring of the GTDBTK classification lineage string.

In [ ]:
TARGET_PHYLUM = "Bacteroidota"  # substitute any taxon substring from the GTDBTK lineage

spark.sql(f"""
    SELECT b2wr.biosample_id,
           bsm.env_broad_scale_term_id,
           bsm.geo_loc_name_has_raw_value,
           COUNT(*) AS n_matching_mags,
           COLLECT_LIST(g.name) AS bin_ids
    FROM   nmdc_results.gtdbtk_bacterial_summary g
    JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
             ON b2wr.workflow_run_id = g.workflow_run_id
    JOIN   nmdc_metadata.biosample_set bsm
             ON bsm.id = b2wr.biosample_id
    WHERE  g.classification LIKE '%{TARGET_PHYLUM}%'
    GROUP BY b2wr.biosample_id, bsm.env_broad_scale_term_id, bsm.geo_loc_name_has_raw_value
    ORDER BY n_matching_mags DESC
    LIMIT 20
""").toPandas()

### 5. CheckM quality threshold → biosamples

Find biosamples that recovered high-quality MAGs (completeness ≥ 90, contamination ≤ 5 —
MIMAG standard). Returns the biosample and the count of high-quality bins.

In [ ]:
spark.sql("""
    SELECT b2wr.biosample_id,
           bsm.env_broad_scale_term_id,
           bsm.geo_loc_name_has_raw_value,
           COUNT(*) AS n_hq_mags,
           ROUND(AVG(c.completeness), 1)  AS avg_completeness,
           ROUND(AVG(c.contamination), 2) AS avg_contamination
    FROM   nmdc_results.checkm_statistics c
    JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
             ON b2wr.workflow_run_id = c.workflow_run_id
    JOIN   nmdc_metadata.biosample_set bsm
             ON bsm.id = b2wr.biosample_id
    WHERE  c.completeness >= 90
      AND  c.contamination <= 5
    GROUP BY b2wr.biosample_id, bsm.env_broad_scale_term_id, bsm.geo_loc_name_has_raw_value
    ORDER BY n_hq_mags DESC
    LIMIT 20
""").toPandas()